<a href="https://colab.research.google.com/github/yuta1-cell/Rice-Leaf-Health-App/blob/main/rice_disease_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import kagglehub

# Downloading latest version
path = kagglehub.dataset_download("anshulm257/rice-disease-dataset")

print("Access to the dataset:", path)

# Setting sizes
batch_size = 32
image_size = (300, 300)

Using Colab cache for faster access to the 'rice-disease-dataset' dataset.
Access to the dataset: /kaggle/input/rice-disease-dataset


In [ ]:
import tensorflow as tf
import os

fixed_path = os.path.join(path, "Rice_Leaf_AUG")

# Training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    fixed_path,
    validation_split = 0.2,
    subset = "training",
    seed = 123,
    image_size = image_size,
    batch_size = batch_size
)

# Validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    fixed_path,
    validation_split = 0.2,
    subset = "validation",
    seed = 123,
    image_size = image_size,
    batch_size = batch_size
)

# Class names
class_names = train_ds.class_names
print("Found classes:", class_names)

Found 3829 files belonging to 6 classes.
Using 3064 files for training.
Found 3829 files belonging to 6 classes.
Using 765 files for validation.
Found classes: ['Bacterial Leaf Blight', 'Brown Spot', 'Healthy Rice Leaf', 'Leaf Blast', 'Leaf scald', 'Sheath Blight']


In [ ]:
# AUTOTUNE
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [6]:
model = tf.keras.applications.MobileNetV2(
    input_shape=(300, 300, 3),
    include_top=False,
    weights='imagenet'
)

# Data augmentation
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.3),
    tf.keras.layers.RandomZoom(0.3),
    tf.keras.layers.RandomContrast(0.3),
])

# Creating models
model = tf.keras.Sequential([
    data_aug,
    tf.keras.layers.Lambda(preprocess_input),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

/tmp/ipykernel_703/1734559514.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2)
]

# Start
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
28/96 ━━━━━━━━━━━━━━━━━━━━ 41s 614ms/step - accuracy: 0.3782 - loss: 1.9938

In [ ]:
model.save('saved_model')